# Murphy — GraphRAG over the flight knowledge graph

**CS 5542 · Challenge 3 · Human–AI Co-Design**

> question → Neo4j retrieval → retrieved context → LLM → grounded answer

Challenge 2 answered *"will I land on time?"* from a table of comparable flights.
Challenge 3 built a knowledge graph of how flights connect. This notebook puts a
natural-language interface on top of it.

**Two rules the system enforces rather than hopes for:**

1. **No Text2Cypher.** Queries are fixed, parameterised templates. Building this graph
   produced three Cypher mistakes that returned *wrong answers rather than errors*, so
   a model generating Cypher has nothing to correct against.
2. **The LLM never produces a value.** Every number in an answer is retrieved from the
   graph, and the evidence is shown beside the answer so it can be checked.

In [1]:
import sys; sys.path.insert(0, "../src")
from murphy import graphrag as g

llm = g._llm()
print("LLM:", g.LLM_MODEL)

LLM: openai/gpt-oss-120b


---
## The graph it runs on

Built in the previous stage: one day of US mainline flying, 19 July 2024.

In [2]:
from murphy import kg
nodes, rels = kg.counts()
print(f"{nodes:,} nodes   {rels:,} relationships")
for r in kg.run("MATCH (n) UNWIND labels(n) AS l RETURN l AS label, count(*) AS n ORDER BY n DESC"):
    print(f"   {r['label']:<14} {r['n']:>7,}")

37,059 nodes   117,489 relationships


   Flight          15,224
   Service         15,158
   Aircraft         3,596
   Weather          2,843
   Airport            202
   PolicyClause        19
   Airline              9
   Traveler             4
   Itinerary            4


---
# Q1 · Basic retrieval

**"What flights are on this itinerary, how late were they, and when did the traveller
need to arrive?"**

A single entity and its immediate neighbours: the trip, its legs, the airports, and
the traveller's own deadline. The `Service` quantiles come along because they are what
makes a delay mean something — 186 minutes late reads differently on a route whose
90th percentile is 94.

In [3]:
r1 = g.retrieve_itinerary("itin-no-escape")
print(r1.cypher)

MATCH (t:Traveler)-[:BOOKED]->(i:Itinerary {itinerary_id: $itinerary_id})
MATCH (i)-[leg:CONTAINS]->(f:Flight)
MATCH (f)-[:DEPARTS_FROM]->(from:Airport)
MATCH (f)-[:ARRIVES_AT]->(to:Airport)
MATCH (f)-[:INSTANCE_OF]->(s:Service)
RETURN leg.leg_order        AS leg,
       f.name               AS flight,
       from.iata            AS from,
       to.iata              AS to,
       f.sched_dep_local    AS scheduled_departure,
       f.sched_arr_local    AS scheduled_arrival,
       f.outcome_arr_delay_min AS arrived_late_by_min,
       s.p50_arr_delay      AS typical_delay_for_this_service,
       s.p90_arr_delay      AS bad_day_delay_for_this_service,
       s.n_flights          AS service_sample_size,
       t.arrive_by_utc      AS traveler_deadline
ORDER BY leg


In [4]:
a1 = g.answer(r1, question="What flights are on this itinerary, how late were they, "
                           "and when did the traveller need to arrive?", llm=llm)
a1.show()


Q: What flights are on this itinerary, how late were they, and when did the traveller need to arrive?
   tool: itinerary   evidence: 2 rows   1.4s

   leg=1 · flight=DL1246 MKE-ATL · from=MKE · to=ATL
       scheduled_departure=14:59 · scheduled_arrival=17:03
       arrived_late_by_min=56 · typical_delay_for_this_service=-9
       bad_day_delay_for_this_service=33 · service_sample_size=208
       traveler_deadline=2024-07-19T22:00:00.000000000-04:00
   leg=2 · flight=DL2498 ATL-EWR · from=ATL · to=EWR
       scheduled_departure=18:48 · scheduled_arrival=20:59
       arrived_late_by_min=186 · typical_delay_for_this_service=6
       bad_day_delay_for_this_service=94 · service_sample_size=310
       traveler_deadline=2024-07-19T22:00:00.000000000-04:00

A: The itinerary includes flight DL1246 MKE‑ATL and flight DL2498 ATL‑EWR.
   DL1246 arrived 56 minutes late, and DL2498 arrived 186 minutes late.   The
   traveller needed to be at the destination
   by 2024‑07‑19T22:00:00.000000000‑04:0

### Evaluation — Q1

**Correct.** Both legs, both delays, and the deadline are reproduced exactly from the
evidence rows. Nothing invented.

**What it missed, and this is the interesting part.** The model reported the deadline
as the raw timestamp `2024-07-19T22:00:00-04:00` and did not notice that leg 2 arrived
186 minutes late against a scheduled 20:59 — which puts the traveller on the ground at
**00:05, two hours past their deadline.** The answer is grounded but not *useful*: it
restates the evidence without drawing the one conclusion the traveller cares about.

Doing that arithmetic would mean the LLM computing, which rule 2 forbids. The right
fix is for the retriever to return `minutes_past_deadline` as a field, so the model can
report it rather than derive it. **Retrieval should compute what the answer needs.**

---
# Q2 · Multi-hop retrieval

**"This flight ran late — what else did the same aircraft fly, and how much of their
delay was blamed on it?"**

This is the multi-hop requirement: `[:PRECEDES*1..4]` follows one airframe forward
through up to four subsequent flights.

In [5]:
r2 = g.retrieve_cascade("2024-07-19_WN4854_LBB_DAL")
print(r2.cypher)

MATCH path = (origin:Flight {flight_id: $flight_id})-[:PRECEDES*1..4]->(later:Flight)
MATCH (ac:Aircraft)-[:FLIES]->(origin)
MATCH (later)-[:DEPARTS_FROM]->(from:Airport)
MATCH (later)-[:ARRIVES_AT]->(to:Airport)
RETURN ac.tail_number AS aircraft,
       length(path)   AS hops_downstream,
       later.name     AS later_flight,
       from.iata      AS from,
       to.iata        AS to,
       later.outcome_arr_delay_min           AS arrived_late_by_min,
       later.outcome_delay_carrier_min       AS blamed_on_carrier_min,
       later.outcome_delay_late_aircraft_min AS blamed_on_late_aircraft_min
ORDER BY hops_downstream


In [6]:
a2 = g.answer(r2, question="This flight ran late. What else did the same aircraft fly, "
                           "and how much of their delay was blamed on it?", llm=llm)
a2.show()


Q: This flight ran late. What else did the same aircraft fly, and how much of their delay was blamed on it?
   tool: cascade   evidence: 4 rows   1.5s

   aircraft=N928WN · hops_downstream=1 · later_flight=WN4854 DAL-MCO · from=DAL
       to=MCO · arrived_late_by_min=-9
   aircraft=N928WN · hops_downstream=2 · later_flight=WN1547 MCO-CMH · from=MCO
       to=CMH · arrived_late_by_min=105 · blamed_on_carrier_min=105
       blamed_on_late_aircraft_min=0
   aircraft=N928WN · hops_downstream=3 · later_flight=WN2637 CMH-HOU · from=CMH
       to=HOU · arrived_late_by_min=115 · blamed_on_carrier_min=2
       blamed_on_late_aircraft_min=106
   aircraft=N928WN · hops_downstream=4 · later_flight=WN2679 HOU-MDW · from=HOU
       to=MDW · arrived_late_by_min=115 · blamed_on_carrier_min=2
       blamed_on_late_aircraft_min=113

A: The N928WN later operated WN1547 MCO‑CMH, with 0 minutes of the delay blamed
   on the aircraft; WN2637 CMH‑HOU, with 106 minutes blamed on the aircraft;
   and WN2679 H

### Evaluation — Q2

**Every figure is correct, but the summary is selective — and that is the finding.**

The three delays and their late-aircraft attributions (0, 106, 113) all match the
evidence exactly. But the model **omitted two things it was given**: it never mentions
the *carrier* figures at all, and it drops the first hop (`WN4854 DAL-MCO`, which
arrived nine minutes early) without saying so.

Those omissions matter. The carrier column is what shows the delay *originated* with
the airline and was then inherited; without it the reader sees three late flights and
no cause. And dropping the clean first leg removes the contrast that makes the break
visible. Nothing is wrong, and the answer is still less informative than the evidence.

**Grounding constrains what a model may say. It does not make it say the useful part.**

**Why the graph was necessary.** Nothing in the flight table expresses the *sequence*
these flights were flown in. They share a carrier, a date and a tail number, but the
ordering is implicit in departure times, so reconstructing the chain in SQL means a
recursive self-join ordered by time. Here it is one variable-length pattern, and the
path *is* the explanation.

**One thing to be careful about.** The delay does not grow along the chain — it is 105,
then 115, then 115. It **persists**. The aircraft fell behind once and never recovered:
its turnarounds *added* 3 and 4 minutes rather than absorbing any, and the legs made up
8, lost 7 and made up 4 minutes in the air. A reader skimming the table could mistake
the sequence for compounding, and the model's phrasing does not guard against that.

---
# Q3 · Analytics

**"Of the Atlanta→Newark options on 19 July, how many were flown by an aircraft already
running late, and how late did they arrive on average?"**

An aggregate across a retrieved set rather than a lookup. Counting happens in Cypher,
precisely so the model reports a number it was handed instead of computing one.

This question is also where the system failed first, so both versions are shown.

In [7]:
# The first version of this retriever returned a single field named
# `avg_arrival_delay_min` = 257, computed across ALL fifteen options.
AMBIGUOUS = '''
MATCH (alt:Flight)-[:DEPARTS_FROM]->(hub:Airport {iata: $hub})
MATCH (alt)-[:ARRIVES_AT]->(dest:Airport {iata: $dest})
OPTIONAL MATCH (upstream:Flight)-[:PRECEDES]->(alt)
WITH alt, upstream IS NOT NULL AND upstream.outcome_arr_delay_min >= 45 AS compromised
RETURN count(alt) AS options,
       sum(CASE WHEN compromised THEN 1 ELSE 0 END) AS aircraft_already_late,
       round(avg(alt.outcome_arr_delay_min)) AS avg_arrival_delay_min
'''
before = g._run("route-risk (ambiguous)", "", AMBIGUOUS, hub="ATL", dest="EWR")
print("EVIDENCE:", before.as_context())
print()
import textwrap
ans = g.answer(before, question="Of the Atlanta to Newark options on 19 July, how many "
       "were flown by an aircraft already running late, and how late did they arrive "
       "on average?", llm=llm).answer
for i, line in enumerate(textwrap.wrap(ans, 76)):
    print(("ANSWER : " if i == 0 else "         ") + line)

EVIDENCE: options=15 · aircraft_already_late=8 · avg_arrival_delay_min=257.0



ANSWER : Eight of the 15 Atlanta‑to‑Newark options on 19 July were flown by an
         aircraft already running late, and those flights arrived on
         average 257 minutes late.


**The failure.** The model answers *"those flights arrived on average 257 minutes
late"* — attaching the average to the eight compromised flights. But 257 is the average
across **all fifteen**. The true average among the eight is 268.

Every number it wrote *was* in the evidence, so rule 2 held. The retriever handed back
an ambiguous label and the model resolved the ambiguity confidently and wrongly.

**The fix: make the field names unambiguous.**

In [8]:
r3 = g.retrieve_route_risk("ATL", "EWR")
print("EVIDENCE:", r3.as_context())
print()
import textwrap
a3 = g.answer(r3, question="Of the Atlanta to Newark options on 19 July, how many were "
                           "flown by an aircraft already running late, and how late did "
                           "they arrive on average?", llm=llm)
for i, line in enumerate(textwrap.wrap(a3.answer, 76)):
    print(("ANSWER : " if i == 0 else "         ") + line)

EVIDENCE: total_options=15 · options_with_traceable_upstream=8
    options_whose_aircraft_was_already_late=8
    avg_delay_across_ALL_options=257.0
    avg_delay_among_COMPROMISED_options=268.0
    avg_delay_among_OTHER_options=212.0



ANSWER : Eight of the Atlanta‑to‑Newark options on 19 July were flown by an aircraft
         that was already late (options_whose_aircraft_was_already_late = 8). Those
         “compromised” options arrived on average 268 minutes late
         (avg_delay_among_COMPROMISED_options = 268.0).


### Evaluation — Q3

**Wrong before the fix, correct after.** The model now reports 268 — the average among
the compromised options — because that is the only field whose name says so.

**The lesson generalises, and it sharpens the project's own rule.** *"The LLM never
produces a number"* is necessary but not sufficient. The retriever must also hand back
numbers that **can only mean one thing**. An ambiguous column is an invitation to
misread, and a fluent model will accept it.

**What the answer shows.** Fifteen ways to get from Atlanta to Newark that day.
`options_with_traceable_upstream=8` and `options_whose_aircraft_was_already_late=8` —
so of the eight whose inbound flight is inside the loaded day, **every one** was flown
by an aircraft already 45+ minutes late. A forecaster scoring each leg independently
sees fifteen separate risks; the graph shows one cause behind them.

The other seven are not evidence of anything: their inbound leg fell outside the single
day the graph holds, so the question simply cannot be asked of them.

---
# Routing · agentic tool selection

The three retrievers are wrapped as tools and an LLM chooses between them — "agentic"
in the sense the lecture uses (slide 9).

**It is tool selection, not verification.** There is no retry, no evidence check and no
abstention. If it picks wrongly, it answers the wrong question.

In [9]:
for q in ["What is on my trip and when do I need to land?",
          "This flight was late -- what else did that plane mess up?",
          "How many Atlanta to Newark options were already compromised?",
          "Am I owed a hotel?"]:
    r = g.route(q, llm=llm)
    print(f"  {r.tool:<12} <- {q}")

  itinerary    <- What is on my trip and when do I need to land?


  cascade      <- This flight was late -- what else did that plane mess up?


  route-risk   <- How many Atlanta to Newark options were already compromised?


  policy       <- Am I owed a hotel?


### The failure mode, demonstrated

An out-of-scope question has no right tool. The router picks one anyway.

In [10]:
q = "What is the weather like in Denver next Tuesday?"

# Same question, eight times, temperature 0.
routes = [g.route(q, llm=llm).tool for _ in range(8)]
print("routed to:", " · ".join(routes))
print("distinct tools chosen for one question:", sorted(set(routes)))

r = g.route(q, llm=llm)
print(f"\nlast run went to '{r.tool}' and returned {len(r)} rows of unrelated data")
print("ANSWER:", g.answer(r, question=q, llm=llm).answer)

routed to: itinerary · itinerary · policy · itinerary · policy · itinerary · itinerary · itinerary
distinct tools chosen for one question: ['itinerary', 'policy']



last run went to 'itinerary' and returned 2 rows of unrelated data


ANSWER: the graph does not contain this


### Evaluation — routing

**Routing is not even stable.** One question, eight runs, temperature zero — and the
model picks two different tools — six runs to one, two to the other. Temperature 0 does not make an LLM deterministic,
and nothing downstream checks the choice, so the tool that answers a question can
change between runs with no input change at all.

**The grounding layer holds regardless.** Whichever wrong tool is chosen, the model is
handed rows that cannot answer the question and says *"the graph does not contain
this"* rather than inventing a forecast. The safety property survives a moving target.

That is the honest shape of it: **routing is unreliable, grounding is what makes the
system safe.** And grounding is a behaviour, not a guarantee — a model less obedient to
the prompt would have answered anyway. A verification step that checked the retrieved
rows actually address the question would turn the behaviour into a property, and that
is the clearest improvement to make next.

One crash worth recording: `route()` originally assumed the model always returns
content. It does not — an empty reply raised `IndexError` on roughly one call in three
during testing. A router that crashes is worse than one that chooses badly, so every
unusable reply now falls back to a named tool.

---
# The GraphRAG hinge

The policy retriever is where the graph and the documents meet, and it is the clearest
demonstration of why this is GraphRAG rather than text search.

Same question, asked with and without the graph's context.

In [11]:
q = "can I get a hotel or a meal"

narrowed = g.retrieve_policy(q, flight_id="2024-07-19_DL2498_ATL_EWR")
facts = narrowed.params["narrowed_by"][0]
print("WITH the graph -- the graph first establishes:")
print(f"    carrier = {facts['carrier']}   delay = {facts['delay_min']} min"
      f"   within_airline_control = {facts['within_airline_control']}")
print("  and those three facts filter the clauses:")
for row in narrowed.rows:
    print(f"    {row['airline']:<18} {row['commitment']:<18} threshold={row['threshold_hours']}")

print("\nWITHOUT the graph -- plain full-text search:")
for row in g.retrieve_policy(q).rows:
    print(f"    {row['airline']:<20} {row['commitment']:<18} threshold={row['threshold_hours']}")

WITH the graph -- the graph first establishes:
    carrier = DL   delay = 186 min   within_airline_control = True
  and those three facts filter the clauses:
    Delta Air Lines    meal               threshold=3
    Delta Air Lines    meal               threshold=3
    Delta Air Lines    hotel              threshold=None
    Delta Air Lines    hotel              threshold=None
    Delta Air Lines    ground_transport   threshold=None

WITHOUT the graph -- plain full-text search:


    JetBlue Airways      meal               threshold=3
    Alaska Airlines      meal               threshold=3
    United Air Lines     meal               threshold=3
    American Airlines    meal               threshold=3
    Allegiant Air        meal               threshold=3


**With the graph**, every clause is Delta's — the carrier that actually operated the
flight — and the three-hour meal commitment appears because the delay was 186 minutes.

**Without it**, the same query returns clauses from five other carriers — Alaska,
Allegiant, American, JetBlue and United — and not one of them is the airline this
traveller is flying. Fluent, grounded in real published documents, and useless.

The graph supplies *carrier*, *cause within the airline's control*, and *delay in
minutes*; those three facts turn a text search into an answer. **The graph narrows, the
index reads.**

In [12]:
# The threshold gate, on two legs of the same trip.
for fid, label in [("2024-07-19_DL2498_ATL_EWR", "186 min late"),
                   ("2024-07-19_DL1246_MKE_ATL", " 56 min late")]:
    got = sorted({c["commitment"] for c in
                  g.retrieve_policy("hotel meal rebooking voucher", flight_id=fid, limit=20).rows})
    print(f"  {label}: {', '.join(got)}")

  186 min late: ground_transport, hotel, meal


   56 min late: ground_transport, hotel


A 186-minute delay qualifies for a meal; a 56-minute delay does not. Same traveller,
same airline, same question — the difference comes from a threshold stored on the
clause and a delay stored on the flight.

---
# Human–AI Co-Design

Two iterations, both recorded as they happened.

### Iteration 1 — embeddings → full-text index

| | |
|---|---|
| **Human Design** | Policy lookup needs semantic search over clause text |
| **AI Recommendation** | `SentenceTransformerEmbeddings` + a Neo4j vector index, following the lecture's own example |
| **Human Evaluation** | The dependency stack would not resolve. `sentence-transformers` 6.1 requires `transformers` 5.x, which failed to import against the installed torch 2.2; pinning `transformers` below 5 then left torch unable to see numpy 2.x. Two rounds of pinning, no resolution. Separately: nineteen clauses do not need an embedding model at all |
| **Revised Implementation** | Neo4j's **native full-text index**. No model, no GPU, runs offline during the demo, and is a Neo4j retrieval method in its own right |

**Accepted** the goal, **rejected** the mechanism. Verified it works: *"is anyone going
to pay me cash"* returns contract text rather than a cash commitment, because no US
carrier offers one.

### Iteration 2 — the ambiguous aggregate

| | |
|---|---|
| **Human Design** | One analytics question, answered from a Cypher aggregate |
| **AI Recommendation** | A compact result: `options`, `aircraft_already_late`, `avg_arrival_delay_min` |
| **Human Evaluation** | Hand-checking the answer caught the model attributing the 257-minute average to the eight compromised flights. The true figure for those eight is 268. The model obeyed the rules; the field name was the problem |
| **Revised Implementation** | Three explicitly scoped averages — `ALL_options`, `COMPROMISED_options`, `OTHER_options`. The model now answers 268 |

**The generalisation:** it is not enough that the LLM never computes. The retriever must
return values that cannot be misread.

---
# Limitations

1. **Routing has no verification.** Tool selection only — no retry, no evidence check,
   no abstention. Out-of-scope questions route somewhere and rely on the grounding
   prompt to refuse.
2. **Three fixed templates.** An unanticipated question routes to the nearest tool.
   Coverage is deliberate and narrow.
3. **One day of data.** Any question about trends, seasons or recent history has
   nothing to retrieve.
4. **Grounded is not the same as useful.** Q1 restates the evidence correctly without
   noticing the traveller missed their deadline. Retrieval must compute what the answer
   needs.
5. **Policy coverage is partial** — the DOT dashboard covers all nine carriers, but
   contract-of-carriage text covers only Delta and Southwest. Of the rest, American and
   United returned 403 and a timeout, Alaska's page is JavaScript-rendered with almost
   no text in the HTML, and no working JetBlue URL was found. The DOT dashboard itself
   had to be saved from a browser because transportation.gov returns 403 to every
   automated client tried.

# Next

Route alternatives are retrieved but not **ranked**: that needs the traveller's deadline
applied to each candidate's projected arrival, which is Challenge 4. The deadline is
already a property on `Traveler`, so the structure is in place.